# Lección 12.2 — Auto-atención: consultas, claves y valores (la biblioteca)

> **Módulo 12 · Atención y Transformers** · Nivel 4 — Avanzado
> ⏱️ Tiempo estimado: 100–120 min (lección con ejercicios intercalados ≈ 75 min · ejercicios finales ≈ 30 min)
> 📋 Requisitos previos: Lección 12.1 (atención), Lección 11.2 (*embeddings*)

### 🎯 Objetivos
Al terminar esta lección podrás:
- Explicar la **auto-atención** con la analogía de la biblioteca: **consulta**, **clave** y **valor**.
- Calcular a mano (numpy) la fórmula completa `softmax(Q·Kᵀ / √d) · V` y comprobarla con PyTorch.
- Explicar por qué se divide entre **√d** y para qué sirve la **máscara causal**.
- Abrir un modelo real (**GPT-2**) y leer sus mapas de atención.

### 🧭 Cómo usar este notebook
| Icono | Significa |
|---|---|
| 📖 | Lee: explicación del concepto |
| ▶️ | Ejecuta la celda y observa el resultado |
| 🔮 | **Predice** antes de ejecutar: escribe tu apuesta y luego compruébala |
| ✋ | **Tu turno**: ejercicio. Las respuestas están ocultas en "👀 Ver respuesta" |
| ✏️ | Modifica un valor en el código y vuelve a ejecutar |

> 💡 No necesitas saber programar. El código está para que **veas** los conceptos en acción; solo tendrás que cambiar números.

## ▶️ Preparación
Ejecuta esta celda una vez. Carga las herramientas, las funciones `softmax` y `mapa_calor` de la Lección 12.1 y las palabras de la frase del gato.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F

np.set_printoptions(precision=2, suppress=True)
plt.rcParams["figure.figsize"] = (6, 4)

def softmax(puntajes, eje=-1):
    """Convierte puntajes en pesos positivos que suman 1 (fila por fila)."""
    e = np.exp(puntajes - np.max(puntajes, axis=eje, keepdims=True))
    return e / e.sum(axis=eje, keepdims=True)

def mapa_calor(M, filas, columnas, titulo="", ax=None, numeros=True):
    ax = ax or plt.gca()
    ax.imshow(M, cmap="Blues")
    ax.set_xticks(range(len(columnas)), columnas, rotation=45, ha="right")
    ax.set_yticks(range(len(filas)), filas)
    if numeros:
        for i in range(M.shape[0]):
            for j in range(M.shape[1]):
                claro = M[i, j] < M.min() + 0.6 * (M.max() - M.min())
                ax.text(j, i, f"{M[i, j]:.2f}", ha="center", va="center", fontsize=8,
                        color="black" if claro else "white")
    ax.set_title(titulo)

# La frase de la Lección 12.1, con vectores [ser vivo, lugar, acción, estado]
palabras = ["gato", "cruzó", "calle", "estaba", "cansado"]
X = np.array([[1.0, 0.0, 0.0, 0.2],
              [0.0, 0.0, 1.0, 0.0],
              [0.0, 1.0, 0.0, 0.0],
              [0.0, 0.0, 0.5, 0.5],
              [0.2, 0.0, 0.0, 1.0]])
print("✅ Todo listo")

---
## 1. 📖 La intuición: la biblioteca

Entras a una biblioteca buscando **"libros sobre dragones"**.

| Papel | En la biblioteca | En la auto-atención |
|---|---|---|
| 🔎 **Consulta** (*query*, **Q**) | Lo que **buscas**: "dragones" | Lo que una palabra **necesita** saber |
| 🏷️ **Clave** (*key*, **K**) | La **etiqueta** del lomo de cada libro: "fantasía, dragones" | Lo que cada palabra **anuncia** que ofrece |
| 📖 **Valor** (*value*, **V**) | El **contenido** del libro | La **información** que cada palabra entrega si la eligen |

El proceso:
1. Comparas tu consulta con **la etiqueta** de cada libro (no con su contenido completo: sería lentísimo).
2. Los libros cuyas etiquetas encajan mejor reciben más atención.
3. Te llevas una **mezcla del contenido** de esos libros, ponderada por qué tanto encajaron.

En la Lección 12.1 cada palabra usaba **el mismo vector** para los tres papeles, y por eso se miraba al espejo. Ahora cada palabra tendrá **tres versiones de sí misma**: una para buscar, otra para anunciarse y otra para entregar información.

"Auto" significa que las consultas, las claves y los valores salen **de la misma frase**: cada palabra busca entre sus propias compañeras.

### ✋ Ejercicio 1 — Encuentra Q, K y V en la vida diaria
En cada situación, identifica cuál es la **consulta**, cuál la **clave** y cuál el **valor**:
1. Buscas una canción en Spotify escribiendo "canciones tristes de los 90".
2. Un reclutador busca "experto en Python" entre 200 currículums.
3. En un diccionario buscas la palabra "efímero".

✍️ *Tus respuestas:*

<details><summary><b>👀 Ver respuesta</b></summary>

| Situación | Consulta (Q) | Clave (K) | Valor (V) |
|---|---|---|---|
| Spotify | "canciones tristes de los 90" | Etiquetas de cada canción (género, año, estado de ánimo) | El audio de la canción |
| Reclutador | "experto en Python" | El titular / palabras clave de cada currículum | La persona completa (su experiencia) |
| Diccionario | "efímero" | La palabra en negrita de cada entrada | La definición |

En todos los casos, **lo que sirve para encontrar** (clave) es distinto de **lo que te llevas** (valor). El diccionario es una búsqueda "dura" (una sola entrada); la atención es una búsqueda "suave" que mezcla varias.
</details>

---
## 2. 📖 El concepto: tres matrices crean los tres papeles

¿Cómo saca cada palabra sus tres versiones? Multiplicando su vector por **tres matrices** distintas, que son los **parámetros que el modelo aprende**:

```
Q = X · W_Q     (lo que busco)
K = X · W_K     (lo que anuncio)
V = X · W_V     (lo que entrego)
```

Para que lo puedas leer, vamos a diseñar **a mano** las matrices con una idea simple:
- Las palabras de **estado** y de **acción** buscan a un **ser vivo** (¿quién está cansado?, ¿quién cruzó?).
- Los **seres vivos** buscan una **acción** (¿qué hizo el gato?).
- Las claves y los valores serán el propio vector de la palabra (`W_K` y `W_V` = identidad).

In [ ]:
#                ser vivo  lugar  acción  estado     ← lo que BUSCA la palabra
W_Q = np.array([[0.0,      0.0,   6.0,    0.0],    # si soy ser vivo, busco una acción
                [0.0,      0.0,   0.0,    0.0],    # si soy lugar, no busco nada
                [6.0,      0.0,   0.0,    0.0],    # si soy acción, busco un ser vivo
                [6.0,      0.0,   0.0,    0.0]])   # si soy estado, busco un ser vivo
W_K = np.eye(4)   # la clave anuncia lo que la palabra ES
W_V = np.eye(4)   # el valor entrega lo que la palabra ES

Q, K, V = X @ W_Q, X @ W_K, X @ W_V
print("Consulta de 'cansado':", Q[4], "  ← busca un ser vivo")
print("Clave de 'gato'      :", K[0], "  ← anuncia: soy un ser vivo")

### 🔮 Ejercicio 2 — Las formas de las matrices
Tenemos **5 palabras**, cada una con un vector de **4** casillas. Antes de ejecutar:
1. ¿Qué forma (filas × columnas) tendrá `Q`?
2. ¿Y la tabla de puntajes `Q · Kᵀ` (cada palabra contra cada palabra)?
3. Si la frase tuviera 1,000 palabras, ¿cambiaría el tamaño de `W_Q`?

✍️ *Mis predicciones:*

In [ ]:
print("X      :", X.shape)
print("W_Q    :", W_Q.shape)
print("Q      :", Q.shape)
print("Q · Kᵀ :", (Q @ K.T).shape)

<details><summary><b>👀 Ver respuesta</b></summary>

1. `Q` es **5 × 4**: una consulta por palabra.
2. `Q · Kᵀ` es **5 × 5**: un puntaje por cada par (quien pregunta, a quién mira).
3. **No.** `W_Q` es 4 × 4 sin importar la longitud de la frase: **los mismos parámetros sirven para frases de cualquier longitud**. Lo que crece (como n², Lección 12.1) es la tabla de puntajes, no el modelo.
</details>

### 📖 La fórmula completa
Ya conoces todas las piezas. La auto-atención de un Transformer es:

```
Atención(Q, K, V) = softmax( Q · Kᵀ / √d ) · V
                             ───────────   ───
                     1. puntuar y escalar   3. mezclar los valores
                    ──────────────────────
                    2. porcentajes por fila
```

Donde **d** es el tamaño de los vectores de consulta/clave (aquí, 4). Por qué se divide entre √d lo verás en la Sección 4.

### 🔮 Ejercicio 3 — ¿Se arregló el espejo?
En la Lección 12.1, "cansado" se miraba sobre todo a sí mismo (36 %) y a "gato" solo un 19 %. Con las nuevas consultas y claves:
1. ¿A quién mirará ahora "cansado"?
2. ¿A quién mirará "gato"?
3. ¿Qué crees que pasará con "calle", que no busca nada (su fila de `W_Q` es de ceros)?

✍️ *Mis predicciones:*

In [ ]:
d = Q.shape[1]
puntajes = Q @ K.T / np.sqrt(d)
pesos = softmax(puntajes)          # softmax por filas
salida = pesos @ V

mapa_calor(pesos, palabras, palabras, "Auto-atención con Q, K y V")
plt.ylabel("quien pregunta (consulta)"); plt.xlabel("a quién mira (clave)"); plt.show()

<details><summary><b>👀 Ver respuesta</b></summary>

1. "cansado" mira a **"gato"** con ≈77 % (antes, 19 %). ¡Problema resuelto!
2. "gato" mira a **"cruzó"** (≈70 %): busca qué acción hizo.
3. "calle" tiene una consulta de ceros → todos sus puntajes valen 0 → reparte **20 % a cada palabra** (como el Ejercicio 9 de la Lección 12.1).

Además, "cruzó" y "estaba" también miran a "gato" (≈81 %): las dos buscan "quién lo hizo". Separar **lo que busco** de **lo que soy** es la idea central de la auto-atención.
</details>

### ✋ Ejercicio 4 — Lee la salida
La salida de cada palabra es la mezcla de valores que recibió. Ejecuta la celda y responde: ¿qué casilla del vector de **"cansado"** creció más respecto a su vector original? ¿Y qué casilla ganó **"gato"**?

In [ ]:
for w, original, nuevo in zip(palabras, X, salida):
    print(f"{w:<8} original {original}  →  después de la atención {nuevo}")

<details><summary><b>👀 Ver respuesta</b></summary>

- "cansado": la casilla **ser vivo** pasó de 0.2 a ≈0.78. Ahora "sabe" de quién habla.
- "gato": ganó en la casilla **acción** (de 0 a ≈0.78). Ahora "sabe" qué hizo.

Cada palabra sale de la capa **enriquecida con el contexto que buscó**. Un Transformer repite esto muchas veces (capa tras capa) y cada vez las palabras saben un poco más de la frase.
</details>

---
## 3. 📖 Cómo aprende: nadie diseña `W_Q` a mano

Nosotros escribimos `W_Q` a mano para poder leerla. En un modelo real:
- `W_Q`, `W_K` y `W_V` empiezan con **números al azar**.
- Durante el entrenamiento (por ejemplo, adivinar la siguiente palabra), el **descenso de gradiente** (Lección 3.3) y la **retropropagación** (Lección 9.4) las ajustan poco a poco.
- Si mirar a "gato" ayuda a adivinar mejor, el gradiente empuja las matrices para que esa atención crezca.

Nadie le dice al modelo "busca el sujeto". **Descubre** qué buscar porque le ayuda a reducir su pérdida.

### 🔮 Ejercicio 5 — Atención con matrices al azar
Antes de entrenar, `W_Q` y `W_K` son aleatorias. ¿Cómo esperas que se vea el mapa de atención: con patrones claros (como el de arriba) o desordenado?

✍️ *Mi predicción:*

In [ ]:
rng = np.random.default_rng(0)
W_Q_azar, W_K_azar = rng.normal(size=(4, 4)), rng.normal(size=(4, 4))   # ✏️ cambia la semilla
pesos_azar = softmax((X @ W_Q_azar) @ (X @ W_K_azar).T / np.sqrt(4))
mapa_calor(pesos_azar, palabras, palabras, "Antes de entrenar: matrices al azar"); plt.show()

<details><summary><b>👀 Ver respuesta</b></summary>

Desordenado: hay casillas oscuras, pero **no significan nada** (por ejemplo, "calle" mira sobre todo a "gato" con ≈46 % y "cansado" se mira a sí mismo con ≈39 %). Si cambias la semilla, el patrón cambia por completo. El significado de los patrones **aparece con el entrenamiento**; lo verás en un Transformer entrenado en la Lección 12.3 y en GPT-2 al final de esta lección.
</details>

---
## 4. 📖 ¿Qué pasa si…? Por qué dividir entre √d

En la Lección 12.1 viste que puntajes grandes vuelven la atención **afilada** (casi todo a una palabra) y eso debilita el aprendizaje.

El problema: en los modelos reales cada vector tiene **64, 128 o más** casillas. Un producto punto suma una multiplicación por casilla, así que con más casillas los puntajes **crecen solos**: su tamaño típico es más o menos **√d**. Dividir entre √d los devuelve a un rango normal.

### 🔮 Ejercicio 6 — Predice el efecto de d
Generamos consultas y claves al azar con distintas dimensiones `d` y medimos:
- qué tan grandes son los puntajes (su desviación estándar),
- el peso **máximo** que recibe una palabra entre 10, **sin** dividir y **dividiendo** entre √d.

¿Qué pasará con el peso máximo **sin dividir** cuando `d` sea 512?

✍️ *Mi predicción:*

In [ ]:
rng = np.random.default_rng(1)
print(f"{'d':>4} | {'tamaño puntajes':>15} | {'peso máx. SIN dividir':>21} | {'peso máx. dividiendo':>20}")
for d in [4, 16, 64, 512]:                         # ✏️
    q = rng.normal(size=(2000, d))                 # 2000 consultas al azar
    k = rng.normal(size=(2000, 10, d))             # cada una con 10 claves al azar
    s = np.einsum("nd,nkd->nk", q, k)              # puntajes
    sin = softmax(s).max(axis=1).mean()
    con = softmax(s / np.sqrt(d)).max(axis=1).mean()
    print(f"{d:>4} | {s.std():>15.1f} | {sin:>21.2f} | {con:>20.2f}")

<details><summary><b>👀 Ver respuesta</b></summary>

- El tamaño de los puntajes crece como √d: ≈2, 4, 8 y 23.
- **Sin dividir**, el peso máximo sube de ≈0.49 a ≈**0.95**: con d = 512 la atención se vuelve casi "todo a una palabra", al azar y antes de aprender nada. Así el modelo apenas puede corregir sus pesos, porque softmax queda "saturado" y sus gradientes son casi cero.
- **Dividiendo entre √d**, el peso máximo se queda estable (≈0.3) sin importar `d`.

Por eso el artículo original del Transformer la llama ***scaled* dot-product attention**: atención por producto punto **escalado**.
</details>

---
## 5. 📖 La máscara causal: no mirar el futuro

Algunos modelos (como GPT) escriben texto **palabra por palabra**. Al entrenarlos para adivinar la siguiente palabra, si cada palabra pudiera mirar a las que vienen **después**, el modelo haría trampa: vería la respuesta.

La solución es una **máscara causal**: antes del softmax, los puntajes hacia el futuro se cambian por **−∞**. Como e^(−∞) = 0, esas casillas reciben **peso 0**.

```
          gato cruzó calle estaba cansado
gato       ✔    ✖     ✖     ✖      ✖       ← "gato" solo se ve a sí mismo
cruzó      ✔    ✔     ✖     ✖      ✖
calle      ✔    ✔     ✔     ✖      ✖
estaba     ✔    ✔     ✔     ✔      ✖
cansado    ✔    ✔     ✔     ✔      ✔       ← la última palabra ve todo
```

### 🔮 Ejercicio 7 — Atención con máscara
Aplica la máscara a nuestra frase. Antes de ejecutar:
1. ¿Qué peso le dará "gato" a sí mismo?
2. "cansado" está al final, así que puede ver todo. ¿Cambiará su fila respecto al mapa sin máscara?
3. "gato" buscaba una acción ("cruzó"), pero ahora no puede verla. ¿Qué hará?

✍️ *Mis predicciones:*

In [ ]:
n = len(palabras)
futuro = np.triu(np.ones((n, n), dtype=bool), k=1)       # True arriba de la diagonal = futuro
puntajes_mascara = np.where(futuro, -np.inf, Q @ K.T / np.sqrt(4))
pesos_mascara = softmax(puntajes_mascara)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
mapa_calor(pesos, palabras, palabras, "Sin máscara (ve todo)", axes[0])
mapa_calor(pesos_mascara, palabras, palabras, "Con máscara causal (no ve el futuro)", axes[1])
plt.tight_layout(); plt.show()

<details><summary><b>👀 Ver respuesta</b></summary>

1. **100 %**: "gato" es la primera palabra; la única que puede ver es a sí misma.
2. **No cambia** (≈77 % a "gato"): la última palabra ya veía todo lo que había antes.
3. Se queda sin opción: su atención a "cruzó" (70 %) desaparece y todo su peso se queda en sí mismo.

El mapa con máscara siempre es un **triángulo**: la parte de arriba a la derecha (el futuro) vale 0. Volverás a ver este triángulo en GPT-2 dentro de un momento y en la Lección 12.4.
</details>

---
## 6. 📖 Lo mismo, en una línea de PyTorch

Todo lo que hicimos a mano ya viene programado (y optimizado) en PyTorch: `F.scaled_dot_product_attention(Q, K, V)`. Comprobemos que da **exactamente** lo mismo que nuestra versión en numpy.

### 🔮 Ejercicio 8
¿Esperas una diferencia **exactamente cero**, o una diferencia diminuta? ¿Por qué podría no ser exactamente cero?

✍️ *Mi predicción:*

In [ ]:
Qt, Kt, Vt = (torch.tensor(m, dtype=torch.float32) for m in (Q, K, V))

salida_torch = F.scaled_dot_product_attention(Qt, Kt, Vt)                 # sin máscara
salida_torch_causal = F.scaled_dot_product_attention(Qt, Kt, Vt, is_causal=True)

print("Diferencia máxima (sin máscara):", np.abs(salida_torch.numpy() - salida).max())
print("Diferencia máxima (causal)     :", np.abs(salida_torch_causal.numpy() - pesos_mascara @ V).max())

<details><summary><b>👀 Ver respuesta</b></summary>

Una diferencia **diminuta** (del orden de 0.0000001), no exactamente cero. Numpy calculó con números de 64 bits y PyTorch con números de 32 bits (`float32`), que tienen menos decimales. Para fines prácticos, **son iguales**: ya entiendes lo que hace esa línea de PyTorch por dentro.
</details>

---
## 7. 📖 Mirando dentro de un modelo real: GPT-2

GPT-2 (OpenAI, 2019) es un modelo de lenguaje con **12 capas** y **12 cabezas** de atención por capa (las cabezas son el tema de la Lección 12.3; por ahora piensa en ellas como 12 atenciones distintas en paralelo). Fue entrenado con texto en **inglés**, así que usaremos una frase en inglés.

Le pedimos que nos devuelva sus pesos de atención con `output_attentions=True`.

> ⏳ La primera vez se descarga el modelo (≈500 MB). En Colab tarda un minuto.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizador = AutoTokenizer.from_pretrained("gpt2")
# attn_implementation="eager" = calcular la atención "a la antigua", para poder devolver los pesos
from transformers.utils import logging
logging.set_verbosity_error(); logging.disable_progress_bar()                   # ocultar avisos técnicos de la librería
gpt2 = AutoModelForCausalLM.from_pretrained("gpt2", attn_implementation="eager").eval()

def atenciones_gpt2(frase):
    entrada = tokenizador(frase, return_tensors="pt")
    with torch.no_grad():
        salida = gpt2(**entrada, output_attentions=True)
    tokens = [t.replace("Ġ", " ") for t in tokenizador.convert_ids_to_tokens(entrada.input_ids[0])]
    A = torch.stack(salida.attentions)[:, 0].numpy()     # forma: (capas, cabezas, tokens, tokens)
    return tokens, A

tokens, A = atenciones_gpt2("The cat did not cross the street because it was too tired")
print("Tokens:", tokens)
print("Forma de las atenciones (capas, cabezas, tokens, tokens):", A.shape)

### 🔮 Ejercicio 9 — ¿A quién mira "it"?
En la capa 4, cabeza 3 (contando desde 0), ¿a qué token crees que mira más el token **" it"**? Y fíjate en la forma del mapa completo: ¿qué esperas ver en la mitad de arriba a la derecha?

✍️ *Mi predicción:*

In [ ]:
CAPA, CABEZA = 4, 3       # ✏️ explora otras: hay 12 capas (0–11) y 12 cabezas (0–11)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
mapa_calor(A[CAPA, CABEZA], tokens, tokens, f"GPT-2 · capa {CAPA}, cabeza {CABEZA}", axes[0], numeros=False)
i_it = tokens.index(" it")
axes[1].barh(tokens[::-1], A[CAPA, CABEZA, i_it][::-1])
axes[1].set_title('¿A dónde mira " it"?')
plt.tight_layout(); plt.show()
print(f'" it" → " cat": {A[CAPA, CABEZA, i_it, tokens.index(" cat")]:.0%}')

<details><summary><b>👀 Ver respuesta</b></summary>

- **" it" mira a " cat" con ≈87 %**. Esta cabeza aprendió, sin que nadie se lo pidiera, algo parecido a "encuentra el sustantivo al que se refiere el pronombre".
- La mitad de arriba a la derecha es **blanca** (peso 0): GPT-2 usa **máscara causal**. Cada token solo mira hacia atrás.

✏️ Si exploras otras cabezas verás patrones muy distintos: algunas miran a la palabra anterior, otras al primer token… Cada cabeza se especializa (Lección 12.3).
</details>

### 🔮 Ejercicio 10 — "tired" vs. "wide"
Cambiamos la última palabra: *"…because it was too **wide**"* (demasiado **ancha**: ahora "it" se refiere a la calle). ¿Cambiará la atención de **" it"** en esta cabeza? Piensa en la máscara causal antes de responder.

✍️ *Mi predicción:*

In [ ]:
tokens_w, A_w = atenciones_gpt2("The cat did not cross the street because it was too wide")
fila_tired = A[CAPA, CABEZA, i_it]
fila_wide = A_w[CAPA, CABEZA, i_it]
print("Fila de ' it' con 'tired':", fila_tired.round(2))
print("Fila de ' it' con 'wide' :", fila_wide.round(2))
print("¿Idénticas?", np.allclose(fila_tired, fila_wide, atol=1e-6))

<details><summary><b>👀 Ver respuesta</b></summary>

**Idénticas.** Cuando GPT-2 procesa " it", la máscara causal le impide ver "tired" o "wide", que vienen después. Para " it", las dos frases son exactamente iguales hasta ese punto.

Esto conecta con el Ejercicio 1 de la Lección 12.1: los humanos resolvemos "¿a qué se refiere?" mirando **también hacia adelante**. Un modelo con máscara causal (un **decodificador**, como GPT) no puede hacerlo en esa posición; tendrá que resolverlo más tarde, cuando procese las palabras siguientes. Un modelo que mira en ambas direcciones (un **codificador**, como BERT) sí puede. Esa diferencia es el tema de la Lección 12.4.
</details>

### 🔮 Ejercicio 11 — El "estacionamiento" de la atención
En la Lección 12.1 viste que la atención **no puede** abstenerse: los pesos siempre suman 1. ¿Dónde crees que pone GPT-2 la atención cuando no le interesa nada en particular? Medimos, para cada capa, el peso **promedio** que todos los tokens le dan al **primer token** ("The").

✍️ *Mi predicción (¿qué porcentaje?):*

In [ ]:
al_primero = A[:, :, 1:, 0].mean(axis=(1, 2))      # promedio sobre cabezas y tokens (sin contar al propio primer token)
plt.bar(range(12), al_primero)
plt.xlabel("capa"); plt.ylabel("peso promedio al primer token"); plt.ylim(0, 1)
plt.title('¿Cuánta atención recibe el primer token ("The")?'); plt.show()
print("Promedio en todas las capas:", f"{al_primero.mean():.0%}")

<details><summary><b>👀 Ver respuesta</b></summary>

¡Muchísima! Alrededor del **27 %** en la primera capa y **más del 70 %** en la mayoría de las capas profundas (≈64 % en promedio). Una palabra tan poco informativa como "The" no merece tanta atención por su contenido.

Es un truco que el modelo aprendió: cuando una cabeza no necesita información en ese momento, "estaciona" su atención en el primer token, cuyo valor casi no modifica nada. Los investigadores lo llaman **sumidero de atención** (*attention sink*). Moraleja (de nuevo): **un peso alto no siempre significa "esto es importante"**.
</details>

---
## 8. 📖 Perillas y errores comunes

| Perilla | Qué controla | Si la **subes** ⬆️ | Si la **bajas** ⬇️ | Valor típico |
|---|---|---|---|---|
| `d` (dimensión de Q y K por cabeza) | Cuántos "rasgos" puede buscar una consulta | Búsquedas más finas, más cálculo | Más barato, menos expresivo | 64 |
| Escala `1/√d` | Nitidez de la atención | (no se toca) | (no se toca) | Siempre `1/√d` |
| Máscara | Qué posiciones puede mirar cada token | — | — | Causal para generar texto; ninguna para comprender |
| Longitud del contexto | Cuántos tokens entran | Más contexto, costo × n² | Más barato | GPT-2: 1,024 tokens |

| 🚨 Síntoma | Causa probable | Qué probar |
|---|---|---|
| La pérdida no baja y la atención es "todo a una palabra" desde el inicio | Olvidaste dividir entre √d | Usar `scaled_dot_product_attention` |
| El modelo de lenguaje tiene una pérdida de entrenamiento casi 0, pero genera basura | Falta la **máscara causal**: está viendo la respuesta | `is_causal=True` (Lección 12.4) |
| Te sale `nan` en la atención | Una fila entera quedó con −∞ (ninguna posición permitida) | Revisar la máscara: cada token debe poder ver al menos una posición |
| "La cabeza X mira a Y, luego Y causa la predicción" | Confundir atención con explicación | Hacer experimentos de ablación (Proyecto del módulo) |

### 🔮 Ejercicio 12 — Una máscara mal hecha
Alguien escribió la máscara con `k=0` en lugar de `k=1`, así que **también tapó la diagonal**. ¿Qué le pasa a la **primera** palabra?

✍️ *Mi predicción:*

In [ ]:
mascara_mala = np.triu(np.ones((n, n), dtype=bool), k=0)     # ✏️ compárala con k=1
with np.errstate(invalid="ignore"):
    pesos_malos = softmax(np.where(mascara_mala, -np.inf, Q @ K.T / 2))
print(pesos_malos)

<details><summary><b>👀 Ver respuesta</b></summary>

La primera fila sale llena de **`nan`** ("no es un número"): la primera palabra no puede mirar a nadie (todo su renglón vale −∞) y softmax intenta dividir 0 entre 0. Un solo `nan` se contagia a todo el modelo durante el entrenamiento (es la fila "mi pérdida es NaN" de la Lección 9.9). La diagonal **siempre** debe quedar permitida.
</details>

---
## 9. 📋 Ficha del mecanismo: auto-atención

| | Auto-atención (*self-attention*) |
|---|---|
| **Qué es** | Atención donde consultas, claves y valores salen de la misma secuencia |
| **Intuición** | La biblioteca: comparas lo que buscas con las etiquetas y te llevas una mezcla del contenido |
| **Fórmula** | `softmax(Q·Kᵀ / √d) · V`, con `Q = X·W_Q`, `K = X·W_K`, `V = X·W_V` |
| **Qué se aprende** | Las matrices `W_Q`, `W_K`, `W_V` (su tamaño no depende de la longitud del texto) |
| **Variantes** | Sin máscara (codificador, BERT) · Con máscara causal (decodificador, GPT) |
| **✅ Fortalezas** | Conecta cualquier par de palabras en un paso · Totalmente paralela · Los mismos parámetros sirven para cualquier longitud |
| **❌ Debilidades** | Costo n² · No conoce el orden de las palabras (Lección 12.3) · Sus pesos se malinterpretan fácilmente |

## 10. 🧠 Resumen en 5 puntos
1. Cada palabra genera tres vectores: **consulta** (lo que busca), **clave** (lo que anuncia) y **valor** (lo que entrega).
2. La fórmula es `softmax(Q·Kᵀ/√d)·V`: puntuar, escalar, convertir a porcentajes y mezclar valores.
3. Dividir entre **√d** evita que la atención se sature cuando los vectores son grandes.
4. La **máscara causal** impide mirar el futuro; es lo que permite entrenar modelos que escriben (GPT).
5. En GPT-2 real hay cabezas con patrones interpretables ("it" → "cat") y trucos aprendidos (el sumidero de atención en el primer token).

---
## 11. 🏋️ Ejercicios finales

### 11.1 Repaso rápido (opción múltiple)

**1.** En la analogía de la biblioteca, la **clave** es…
- a) lo que buscas · b) la etiqueta de cada libro · c) el contenido del libro · d) el bibliotecario

**2.** Si una frase tiene 20 tokens y `d = 64`, ¿qué forma tiene la tabla de pesos de atención (de una cabeza)?
- a) 20 × 64 · b) 64 × 64 · c) 20 × 20 · d) 64 × 20

**3.** ¿Por qué se divide entre √d?
- a) Para que los pesos sumen 1 · b) Para que los puntajes no crezcan con d y saturen el softmax · c) Para ahorrar memoria · d) Para tapar el futuro

**4.** En un mapa de atención con máscara causal, la casilla (fila 2, columna 5) vale…
- a) 1 · b) 0 · c) depende de la frase · d) −∞

**5.** En GPT-2, las atenciones que devuelve `output_attentions=True` tienen forma (12, 12, n, n) por frase. ¿Qué significa el segundo 12?
- a) Capas · b) Cabezas · c) Tokens · d) Dimensión

✍️ *Mis respuestas:* 1 → _ · 2 → _ · 3 → _ · 4 → _ · 5 → _

<details><summary><b>👀 Ver respuestas</b></summary>

1. **b)** La clave es lo que cada elemento anuncia para ser encontrado.
2. **c)** Un peso por cada par de tokens: 20 × 20. El 64 es el tamaño de cada vector.
3. **b)** El tamaño típico del producto punto crece como √d.
4. **b)** La fila 2 no puede mirar a la posición 5 (es futuro): su peso es 0 (el −∞ estaba **antes** del softmax).
5. **b)** (capas, **cabezas**, tokens que preguntan, tokens a los que miran).
</details>

### 11.2 Preguntas para pensar (sin ejecutar)

**A.** Si `W_K = W_Q`, la consulta y la clave de cada palabra serían iguales. ¿Qué problema de la Lección 12.1 volvería a aparecer?

**B. Caso 💼** Un equipo quiere un modelo que lea una reseña completa de un restaurante y diga si es positiva o negativa. Un compañero propone usar máscara causal "porque así lo hace GPT". ¿Estás de acuerdo? ¿Qué se pierde con la máscara en esta tarea?

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuestas</b></summary>

**A.** El del **espejo**: cada palabra se compara consigo misma usando el mismo vector, y el producto punto de un vector consigo mismo suele ser el más alto. Cada palabra tendería a mirarse a sí misma.

**B.** No hace falta. Aquí la reseña completa está disponible desde el inicio y **no hay que generar** nada palabra por palabra. Con máscara, las primeras palabras no pueden ver el resto: en *"No es que sea malo, es que es **increíble**"*, "malo" no podría ver "increíble". Para **clasificar** un texto completo conviene ver en ambas direcciones (un codificador tipo BERT, Lección 12.4).
</details>

### 11.3 🧩 Práctica integradora: auto-atención a mano, de principio a fin

Vas a calcular una auto-atención **completa** para una frase de 3 palabras: *"yo amo tacos"*. Los números son pequeños a propósito para que puedas hacer cada paso en papel antes de ejecutarlo.

**Paso 1 ▶️ — Los datos**

In [ ]:
frase = ["yo", "amo", "tacos"]
X3 = np.array([[1.0, 0.0],
               [0.0, 1.0],
               [1.0, 1.0]])
W_Q3 = np.array([[1.0, 0.0],
                 [0.0, 2.0]])
W_K3 = np.array([[0.0, 1.0],
                 [1.0, 0.0]])
W_V3 = np.array([[1.0, 0.0],
                 [0.0, 1.0]])
print("✅ Listo")

**Paso 2 🔮 — Consultas y claves.** Calcula en papel `Q = X · W_Q` y `K = X · W_K`. (Pista: `W_K` intercambia las dos casillas de cada vector.)

✍️ *Mis Q y K:*

In [ ]:
Q3, K3, V3 = X3 @ W_Q3, X3 @ W_K3, X3 @ W_V3
print("Q =\n", Q3)
print("K =\n", K3)

<details><summary><b>👀 Ver respuesta</b></summary>

- `Q` = [[1, 0], [0, 2], [1, 2]]: `W_Q` duplica la segunda casilla.
- `K` = [[0, 1], [1, 0], [1, 1]]: `W_K` intercambia las casillas.
</details>

**Paso 3 🔮 — Puntajes escalados.** Calcula `Q · Kᵀ` en papel (9 productos punto) y divide entre √2 ≈ 1.41. ¿Qué palabra recibe el puntaje más alto en la fila de **"tacos"**?

✍️ *Mis puntajes:*

In [ ]:
puntajes3 = Q3 @ K3.T / np.sqrt(2)
print("Q·Kᵀ =\n", Q3 @ K3.T)
print("Q·Kᵀ / √2 =\n", puntajes3)

<details><summary><b>👀 Ver respuesta</b></summary>

`Q·Kᵀ` = [[0, 1, 1], [2, 0, 2], [2, 1, 3]]. Dividido entre √2: [[0, 0.71, 0.71], [1.41, 0, 1.41], [1.41, 0.71, 2.12]].

En la fila de "tacos", el puntaje más alto (3 → 2.12) es **"tacos" consigo mismo**.
</details>

**Paso 4 ▶️ — Pesos y salida.** Aplica softmax por filas y mezcla los valores.

In [ ]:
pesos3 = softmax(puntajes3)
salida3 = pesos3 @ V3
mapa_calor(pesos3, frase, frase, "Auto-atención de 'yo amo tacos'"); plt.show()
print("Salida =\n", salida3)

✋ "yo" reparte su atención entre "amo" y "tacos" por igual. ¿Por qué? ¿Cuánto peso le da a cada una?

✍️ *Mi respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

Porque sus puntajes con "amo" y con "tacos" son **iguales** (0.71 los dos), y puntajes iguales dan pesos iguales: ≈**0.40** a cada una y ≈0.20 a sí mismo. Del mismo modo, "amo" reparte ≈0.44 / 0.11 / 0.44 entre "yo", sí mismo y "tacos".
</details>

**Paso 5 ✏️ — Agrega la máscara causal.** Completa la celda: cambia `None` por la máscara correcta (copia la idea de la Sección 5) y predice primero cómo quedará la **primera fila**.

In [ ]:
mascara3 = np.triu(np.ones((3, 3), dtype=bool), k=1)    # ✏️ ya está completa; compruébala
pesos3_causal = softmax(np.where(mascara3, -np.inf, puntajes3))
print(pesos3_causal)

<details><summary><b>👀 Ver respuesta</b></summary>

- Fila "yo": **[1, 0, 0]**: solo se ve a sí mismo.
- Fila "amo": puntajes permitidos 1.41 (yo) y 0 (amo) → ≈**[0.80, 0.20, 0]**.
- Fila "tacos": igual que sin máscara, porque es la última palabra.
</details>

**Paso 6 ▶️ — Compruébalo con PyTorch.** Una sola línea debe reproducir tus pesos causales y tu salida.

In [ ]:
salida3_torch = F.scaled_dot_product_attention(
    torch.tensor(Q3, dtype=torch.float32), torch.tensor(K3, dtype=torch.float32),
    torch.tensor(V3, dtype=torch.float32), is_causal=True)
print("PyTorch :\n", salida3_torch.numpy())
print("A mano  :\n", pesos3_causal @ V3)

✋ ¿Coinciden? Si es así, acabas de programar **a mano** el corazón de GPT. 🎉

<details><summary><b>👀 Ver respuesta</b></summary>

Coinciden (salvo diferencias diminutas por usar 32 bits). Por ejemplo, la salida de "amo" es ≈[0.80, 0.20]: 80 % del valor de "yo" ([1, 0]) más 20 % de su propio valor ([0, 1]).
</details>

---
## 📚 Para profundizar
- 📄 **Artículo:** Vaswani y colegas (2017), *Attention Is All You Need*: el artículo del Transformer (Sección 3.2).
- 🎮 **Interactivo:** *Transformer Explainer* (poloclub.github.io/transformer-explainer): GPT-2 funcionando en el navegador, con sus mapas de atención.
- 🛠️ **Herramienta:** *BertViz* (biblioteca de Python) para explorar las cabezas de atención de GPT-2 y BERT (para Colab: `pip install bertviz`).
- 📘 **Blog:** Jay Alammar, *The Illustrated Transformer* (inglés).

---
⬅️ **Anterior:** [12.1 Atención: "¿a qué palabras debo prestar atención?"](12.1-atencion.ipynb) · ➡️ **Siguiente:** [12.3 Multi-cabeza, codificación posicional y el bloque Transformer](12.3-multicabeza-posicion-bloque.ipynb)